# Model 2: Fine-Tuned Gemma-2-2b-it with QLoRA

**Author:** Berkay Kaya (Team 11)  
**Course:** Applications of Data Science -- SBWL Data Science, WU Wien, SS26

## Overview
This notebook implements a supervised fine-tuning pipeline for Austrian tax law question answering in German. The base model `google/gemma-2-2b-it` is adapted using QLoRA, which combines 4-bit quantization with LoRA adapters to make fine-tuning feasible on limited GPU resources. The training data consists of rule-based instruction-answer pairs generated from Austrian tax law documents, with the goal of teaching the model domain-specific legal language and citation patterns without using external API calls.

## Architecture
1. **PDF Ingestion:** Extract text from Austrian tax law PDFs (EStG, KStG, UStG) using PyMuPDF  
2. **Section Parsing:** Split legal texts into paragraph-based sections and clean the extracted content  
3. **Synthetic Training Data Generation:** Create instruction-answer pairs from legal sections using rule-based templates  
4. **Chat Formatting:** Convert the examples into the Gemma instruction format  
5. **Fine-Tuning:** Train `google/gemma-2-2b-it` with QLoRA (4-bit quantization + LoRA adapters)  
6. **Inference:** Generate answers for all 643 questions from `dataset_clean.csv` and save the results to CSV  

## Requirements
- Kaggle notebook with **GPU T4 x2** and **Internet** enabled  
- Input dataset: `austrian-tax-laws` (contains `EStG.pdf`, `KStG.pdf`, `UStG.pdf`, `dataset_clean.csv`)  
- Hugging Face access token for downloading `google/gemma-2-2b-it`  

## Cell 1 — Install Packages
All required Python packages for fine-tuning and inference.

- `import sys` loads Python's `sys` module, which provides information about the current Python interpreter.
- `sys.executable` returns the exact path of the Python executable that is currently running the notebook.
- `!{...}` tells Jupyter to run the command as a shell command instead of normal Python code.
- `transformers` is the Hugging Face library for loading pretrained models and tokenizers.
- `accelerate` helps manage GPU-based training and is used internally by `transformers`.
- `bitsandbytes` enables 4-bit quantization, which reduces memory usage significantly.
- `peft` provides parameter-efficient fine-tuning methods such as LoRA.
- `trl` includes `SFTTrainer`, which is used for supervised fine-tuning.
- `datasets` provides efficient dataset objects for model training.
- `PyMuPDF` is used to read PDF files and is later imported as `fitz`.
- `pandas` is used for CSV and tabular data handling.
- `-q` means “quiet” and suppresses long installation logs.

### Project Impact
Installing the libraries in the same Python environment as the notebook avoids version mismatches later. This cell is essential because the whole model 2 pipeline depends on these packages being available.


In [ ]:
import sys
!{sys.executable} -m pip install -q transformers accelerate bitsandbytes peft trl datasets PyMuPDF pandas

## Cell 2 — Imports
Modules that are needed for preprocessing, training, and inference.

- `import os` gives access to operating system functions, such as environment variables and file paths.
- `os.environ` is a dictionary of the system’s environment variables.
- `CUDA_VISIBLE_DEVICES = "0"` tells CUDA to use only GPU 0 and ignore any other GPUs.
- This matters because QLoRA with `bitsandbytes` can be unstable in multi-GPU Kaggle environments.
- `import re` loads Python’s regular expression module for pattern-based text processing.
- `import fitz` loads PyMuPDF under its internal package name so PDF files can be read page by page.
- `Dataset` from Hugging Face is an optimized format for training data.
- `AutoTokenizer` automatically loads the correct tokenizer for the chosen model.
- `AutoModelForCausalLM` loads a text generation model.
- `BitsAndBytesConfig` defines how 4-bit quantization should be applied.
- `LoraConfig` defines the LoRA setup.
- `get_peft_model` attaches LoRA adapters to the base model.
- `prepare_model_for_kbit_training` prepares a quantized model for training.
- `SFTTrainer` manages supervised fine-tuning.
- `SFTConfig` stores the training configuration.

### Project Impact
This cell sets the technical foundation for the notebook. Without these imports, the project could neither preprocess the legal sources nor fine-tune Gemma efficiently on limited GPU memory.


In [2]:
import os
# Force single GPU — QLoRA + bitsandbytes does not work with multi-GPU DataParallel
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import re
import random
import fitz  # PyMuPDF
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print("All imports successful.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")


All imports successful.
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Cell 3 — Configuration
This cell defines all important settings for the model, file locations, and training process.

- `MODEL_NAME = "google/gemma-2-2b-it"` selects Gemma 2 with 2 billion parameters in the instruction-tuned version.
- The `it` suffix means the model already understands chat-like instruction formats.
- `PDF_DIR` points to the folder containing the uploaded PDF files on Kaggle.
- `DATASET_CSV` points to the cleaned dataset with prompts.
- `OUTPUT_CSV` is the final file for generated answers.
- `CHECKPOINT_FILE` stores intermediate inference progress so work is not lost if the session stops.
- `SAVE_DIR` is the directory where the fine-tuned model will be saved.
- `/kaggle/input/` is read-only and contains uploaded input files.
- `/kaggle/working/` is writable and is used for outputs, checkpoints, and saved models.
- `LORA_R = 16` sets the LoRA rank, meaning only a small number of trainable parameters are added.
- `LORA_ALPHA = 32` scales the effect of the LoRA adapters.
- `TARGET_MODULES = ["q_proj", "v_proj"]` means LoRA is applied to the query and value projection layers in the attention mechanism.
- `NUM_EPOCHS = 3` means the model will see the training data three times.
- `LEARNING_RATE = 2e-4` controls how strongly the model weights are updated during training.
- `BATCH_SIZE = 4` means four examples are processed at once on the GPU.
- `GRAD_ACCUM_STEPS = 4` simulates a larger effective batch by accumulating gradients across four mini-batches.

### Project Impact
This configuration cell directly controls model quality, runtime, and memory usage. Good settings here make the project feasible on free Kaggle GPUs while still allowing useful adaptation to Austrian tax law.


In [3]:
# --- Model ---
MODEL_NAME = "google/gemma-2-2b-it"

# --- Paths (Kaggle) ---
PDF_DIR = "/kaggle/input/datasets/berkaya462/a-o-ds-4/"
DATASET_CSV = "/kaggle/input/datasets/berkaya462/a-o-ds-4/dataset_clean.csv"
OUTPUT_CSV = "/kaggle/working/model2_finetuned_results.csv"
CHECKPOINT_FILE = "/kaggle/working/model2_checkpoint.json"
SAVE_DIR = "/kaggle/working/model2_gemma_qlora"

# --- PDF file names (must match exactly what is on Kaggle) ---
PDF_FILES = {
    "EStG": "EStG.pdf",
    "KStG": "KStG 1988 Fassung vom 03.04.2026.pdf",
    "UStG": "UStG.pdf",
}

# --- LoRA hyperparameters ---
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

# --- Training hyperparameters ---
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
MAX_SEQ_LENGTH = 512

# --- Inference ---
MAX_NEW_TOKENS = 300
REPETITION_PENALTY = 1.15

print("Configuration set.")
# Verify PDF files exist
for name, filename in PDF_FILES.items():
    path = os.path.join(PDF_DIR, filename)
    exists = os.path.exists(path)
    print(f"  {name}: {'✓' if exists else '✗ NOT FOUND'} — {path}")

Configuration set.
  EStG: ✓ — /kaggle/input/datasets/berkaya462/a-o-ds-4/EStG.pdf
  KStG: ✓ — /kaggle/input/datasets/berkaya462/a-o-ds-4/KStG 1988 Fassung vom 03.04.2026.pdf
  UStG: ✓ — /kaggle/input/datasets/berkaya462/a-o-ds-4/UStG.pdf


## Cell 4 — Hugging Face Login
Authenticates the notebook with Hugging Face so the Gemma model can be downloaded.

- `UserSecretsClient` is a Kaggle-specific utility for securely accessing saved secrets.
- `.get_secret("HF_TOKEN")` reads the Hugging Face access token from Kaggle’s secret storage.
- `login(token=secret)` logs in to Hugging Face using that token.
- This step is required because Gemma is a gated model and cannot be downloaded anonymously.
- You must accept the Gemma license at https://huggingface.co/google/gemma-2-2b-it before running this.

### Project Impact
This cell is important because it unlocks access to the base model used for fine-tuning. Without it, the notebook cannot load Gemma and the training pipeline stops immediately.

In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Load HF token from Kaggle Secrets and log in
secret = UserSecretsClient().get_secret("HF_TOKEN")
login(token=secret)
print("HuggingFace login successful.")

HuggingFace login successful.


## Cell 5 — Read PDF Files
This cell defines a function that extracts plain text from a PDF file.

- `fitz.open(pdf_path)` opens the PDF document.
- `doc` is the PDF object returned by PyMuPDF.
- `text = ""` creates an empty string that will hold the full document text.
- `for page in doc:` loops through the PDF one page at a time.
- `page.get_text()` extracts the text content from the current page.
- `text += ...` appends each page’s text to the full document string.
- `doc.close()` closes the file and frees memory.
- `return text` returns the full extracted text for later processing.

### Project Impact
This function turns legal PDFs into raw text that the project can actually work with. It is the bridge between the law documents and the synthetic training examples used for fine-tuning.


In [5]:
def extract_pdf_text(pdf_path):
    """Extract all text from a PDF file using PyMuPDF."""
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

# Extract text from all three PDFs
raw_texts = {}
for law_name, filename in PDF_FILES.items():
    path = os.path.join(PDF_DIR, filename)
    raw_texts[law_name] = extract_pdf_text(path)
    print(f"{law_name}: {len(raw_texts[law_name]):,} characters extracted")

print(f"\nTotal PDFs processed: {len(raw_texts)}")

EStG: 948,838 characters extracted
KStG: 214,212 characters extracted
UStG: 312,566 characters extracted

Total PDFs processed: 3


## Cell 6 — Split Text into § Sections
This cell splits the extracted legal text into paragraph-based sections.

- `re.split(...)` divides the full text wherever the specified pattern appears.
- `§\s*\d+[a-z]?` matches a legal paragraph reference such as `§ 7` or `§ 7a`.
- `§` matches the paragraph sign.
- `\s*` means optional whitespace.
- `\d+` means one or more digits.
- `[a-z]?` means an optional lowercase letter.
- The parentheses around the pattern ensure that the separator itself is kept in the result.
- `range(1, len(parts) - 1, 2)` iterates through the split result in steps of two, because the output alternates between section references and section text.
- `section_ref = parts[i].strip()` stores the legal reference.
- `section_text = parts[i + 1].strip()` stores the corresponding paragraph text.
- `re.sub(r"\s+", " ", section_text)` replaces repeated spaces and line breaks with single spaces.
- The length check truncates very long sections to 2000 characters to keep memory usage manageable.

### Project Impact
This preprocessing step creates smaller, cleaner legal units for later question-answer generation. It improves training quality because the model learns from focused sections instead of overly long and messy raw PDF text.


In [6]:
def split_into_sections(text, law_name):
    """Split law text into sections based on § markers.
    Returns list of dicts with 'ref' (e.g. '§ 2 EStG') and 'text'."""

    # Split at § followed by a number
    parts = re.split(r"(§\s*\d+[a-z]?)", text)

    sections = []
    for i in range(1, len(parts) - 1, 2):
        section_ref = parts[i].strip()           # e.g. "§ 2"
        section_text = parts[i + 1].strip()      # the text after it

        # Clean up: remove excessive whitespace
        section_text = re.sub(r"\s+", " ", section_text)

        # Skip very short sections (likely empty or just a title)
        if len(section_text) < 50:
            continue

        # Truncate very long sections to keep training data manageable
        if len(section_text) > 2000:
            section_text = section_text[:2000]

        sections.append({
            "ref": f"{section_ref} {law_name}",  # e.g. "§ 2 EStG"
            "text": section_text,
            "law": law_name,
        })

    return sections

# Split all PDFs into sections
all_sections = []
for law_name, text in raw_texts.items():
    sections = split_into_sections(text, law_name)
    all_sections.extend(sections)
    print(f"{law_name}: {len(sections)} sections extracted")

print(f"\nTotal sections: {len(all_sections)}")
# Show a sample
print(f"\nSample section: {all_sections[0]['ref']}")
print(f"Text preview: {all_sections[0]['text'][:200]}...")

EStG: 2550 sections extracted
KStG: 580 sections extracted
UStG: 513 sections extracted

Total sections: 3643

Sample section: § 1 EStG
Text preview: . Persönliche Steuerpflicht 2. TEIL SACHLICHE STEUERPFLICHT 1. ABSCHNITT...


## Cell 7 — Generate Q&A Pairs
Creating synthetic training examples from the extracted legal sections.

- `TEMPLATES` contains different question-answer patterns.
- `{ref}` and `{text}` are placeholders that are later filled with a legal reference and its content.
- This allows one legal paragraph to produce several slightly different instruction examples.
- `answer_text = text[:500].strip()` shortens the answer to about 500 characters.
- `.strip()` removes unnecessary spaces at the beginning and end.
- `.rfind(".")` searches for the last period in the shortened text.
- If a sentence end is found far enough into the text, the answer is cut there so it ends cleanly instead of in the middle of a sentence.
- `random.sample(TEMPLATES, n_templates)` selects a fixed number of distinct templates without repetition.
- `random.seed(42)` makes the random process reproducible, meaning the same templates are selected each run.

### Project Impact
This cell turns raw law text into instruction-style examples that Gemma can learn from. It has a strong impact on the project because the quality and diversity of these generated Q&A pairs directly shape the fine-tuned model’s behavior.


In [7]:
# Question templates — more diverse = better generalization
TEMPLATES = [
    ("Was regelt {ref}?",
     "Gemäß {ref} gilt Folgendes: {text}"),

    ("Welche Voraussetzungen nennt {ref}?",
     "Die Voraussetzungen nach {ref} sind: {text}"),

    ("Wie ist die Regelung nach {ref}?",
     "Nach {ref} ist geregelt: {text}"),

    ("Wann gilt {ref}?",
     "{ref} findet Anwendung in folgenden Fällen: {text}"),

    ("Was besagt {ref} im Detail?",
     "{ref} besagt: {text}"),

    ("Erkläre {ref}.",
     "{ref} regelt Folgendes: {text}"),

    ("Was sind die Rechtsfolgen nach {ref}?",
     "Die Rechtsfolgen gemäß {ref} sind: {text}"),

    ("Für wen gilt {ref}?",
     "Der Anwendungsbereich von {ref} umfasst: {text}"),

    ("Welche Pflichten ergeben sich aus {ref}?",
     "Aus {ref} ergeben sich folgende Pflichten: {text}"),

    ("Was sind die wichtigsten Punkte in {ref}?",
     "Die wesentlichen Punkte von {ref} sind: {text}"),
]

def generate_qa_pairs(sections, max_pairs=400):
    """Generate Q&A pairs from law sections using templates."""
    qa_pairs = []

    for section in sections:
        ref = section["ref"]
        text = section["text"]

        # Use a shortened version of text for the answer (max ~500 chars)
        answer_text = text[:500].strip()
        # Try to end at a sentence boundary
        last_dot = answer_text.rfind(".")
        if last_dot > 100:
            answer_text = answer_text[:last_dot + 1]

        # Pick 2-3 random templates per section for diversity
        n_templates = min(3, len(TEMPLATES))
        chosen = random.sample(TEMPLATES, n_templates)

        for q_template, a_template in chosen:
            question = q_template.format(ref=ref)
            answer = a_template.format(ref=ref, text=answer_text)
            qa_pairs.append({"question": question, "answer": answer})

    # Shuffle and limit
    random.shuffle(qa_pairs)
    if len(qa_pairs) > max_pairs:
        qa_pairs = qa_pairs[:max_pairs]

    return qa_pairs

# Set seed for reproducibility
random.seed(42)

# Generate pairs — 400 for better coverage (training ~25-30 min on T4)
qa_pairs = generate_qa_pairs(all_sections, max_pairs=400)
print(f"Generated {len(qa_pairs)} Q&A pairs")

# Show a few examples
for i, pair in enumerate(qa_pairs[:3]):
    print(f"\n--- Example {i+1} ---")
    print(f"Q: {pair['question']}")
    print(f"A: {pair['answer'][:150]}...")

Generated 400 Q&A pairs

--- Example 1 ---
Q: Was sind die wichtigsten Punkte in § 3 EStG?
A: Die wesentlichen Punkte von § 3 EStG sind: Abs. 1 Z 10 in der Fassung des Bundesgesetzes BGBl. I Nr. 161/2005 ist anzuwenden, wenn – die Einkommensteu...

--- Example 2 ---
Q: Was sind die wichtigsten Punkte in § 3 EStG?
A: Die wesentlichen Punkte von § 3 EStG sind: und 69, BGBl. Nr. 400/1988) Artikel I ist anzuwenden, 1. wenn die Einkommensteuer veranlagt wird, erstmalig...

--- Example 3 ---
Q: Was sind die wichtigsten Punkte in § 1 EStG?
A: Die wesentlichen Punkte von § 1 EStG sind: des Körperschaftsteuergesetzes 1988 5% der Aufwendungen. Wird die Sonderprämie geltend gemacht, kann für da...


## Cell 8 — Gemma Chat Format
This cell formats each training example in the chat style expected by Gemma.

- `format_chat(question, answer)` creates one formatted conversation example.
- `<start_of_turn>user` marks the beginning of the user message.
- `<end_of_turn>` marks the end of a turn.
- `<start_of_turn>model` marks the beginning of the model answer.
- `SYSTEM_MSG` is inserted before the question so the model also learns the intended role and behavior.
- The function returns one complete training string in Gemma-compatible format.

### Project Impact
This formatting is crucial because instruction-tuned models learn best when examples match their native chat structure. It improves the project by making the fine-tuning data consistent with the model’s expected input style.

In [8]:
SYSTEM_MSG = "Du bist Experte für österreichisches Steuerrecht."

def format_chat(question, answer):
    """Format a Q&A pair in Gemma chat template."""
    return (
        f"<start_of_turn>user\n{SYSTEM_MSG}\n{question}<end_of_turn>\n"
        f"<start_of_turn>model\n{answer}<end_of_turn>"
    )

# Format all pairs
formatted_data = []
for pair in qa_pairs:
    text = format_chat(pair["question"], pair["answer"])
    formatted_data.append({"text": text})

# Show one example
print("--- Formatted training example ---")
print(formatted_data[0]["text"])

--- Formatted training example ---
<start_of_turn>user
Du bist Experte für österreichisches Steuerrecht.
Was sind die wichtigsten Punkte in § 3 EStG?<end_of_turn>
<start_of_turn>model
Die wesentlichen Punkte von § 3 EStG sind: Abs. 1 Z 10 in der Fassung des Bundesgesetzes BGBl. I Nr. 161/2005 ist anzuwenden, wenn – die Einkommensteuer veranlagt wird, erstmalig bei der Veranlagung für das Kalenderjahr 2006. – die Einkommensteuer (Lohnsteuer) durch Abzug eingehoben oder durch Veranlagung festgesetzt wird, erstmalig für Lohnzahlungszeiträume, die nach dem 31. Dezember 2005 enden. 129.<end_of_turn>


## Cell 9 — Create Hugging Face Dataset
Converts the prepared training examples into a Hugging Face dataset object.

- `formatted_data` is a normal Python list containing the formatted chat examples.
- `Dataset.from_list(...)` transforms that list into a Hugging Face `Dataset`.
- This format is optimized for batching, tokenization, and efficient access during training.

### Project Impact
This cell prepares the data in the format required by `SFTTrainer`. It matters for the project because efficient data handling makes the fine-tuning process faster and more stable.


In [9]:
# Create a HuggingFace Dataset from our formatted data
train_dataset = Dataset.from_list(formatted_data)
print(f"Training dataset: {train_dataset}")
print(f"Number of examples: {len(train_dataset)}")
print(f"\nSample entry:\n{train_dataset[0]['text'][:300]}...")

Training dataset: Dataset({
    features: ['text'],
    num_rows: 400
})
Number of examples: 400

Sample entry:
<start_of_turn>user
Du bist Experte für österreichisches Steuerrecht.
Was sind die wichtigsten Punkte in § 3 EStG?<end_of_turn>
<start_of_turn>model
Die wesentlichen Punkte von § 3 EStG sind: Abs. 1 Z 10 in der Fassung des Bundesgesetzes BGBl. I Nr. 161/2005 ist anzuwenden, wenn – die Einkommensteue...


## Cell 10 — Load the Model with 4-Bit Quantization
Loads the base Gemma model in a memory-efficient 4-bit format.

- `BitsAndBytesConfig(...)` defines the quantization setup.
- `load_in_4bit=True` stores the model weights in 4-bit instead of full precision.
- This reduces memory usage dramatically and makes training possible on smaller GPUs.
- `bnb_4bit_quant_type="nf4"` uses the NormalFloat4 quantization format, which works well for normally distributed weights.
- `bnb_4bit_compute_dtype=torch.float16` means computations are still performed in `float16` for better numerical stability.
- `bnb_4bit_use_double_quant=True` applies an additional level of quantization to save even more memory.
- `AutoModelForCausalLM.from_pretrained(...)` loads the pretrained causal language model.
- `device_map={"": 0}` puts the full model on GPU 0.
- `torch_dtype=torch.float16` uses half precision during loading and computation.
- `prepare_model_for_kbit_training(model)` prepares the quantized model for low-bit fine-tuning.
- This usually freezes base weights and activates memory-saving mechanisms such as gradient checkpointing.

### Project Impact
This cell is one of the most important parts of the project because it makes QLoRA feasible on free hardware. Without 4-bit loading, Gemma would likely exceed the available GPU memory.


In [10]:
# 4-bit quantization config for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Force single GPU — QLoRA does not work with multi-GPU DataParallel
# device_map={"": 0} puts all layers on GPU 0
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model dtype: {model.dtype}")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Model loaded: google/gemma-2-2b-it
Model dtype: torch.float32


## Cell 11 — Load the Tokenizer
Loads the tokenizer that converts text into token IDs and back.

- `AutoTokenizer.from_pretrained(MODEL_NAME)` loads the tokenizer that matches the Gemma model.
- The tokenizer converts raw text into numerical IDs that the model can process.
- `tokenizer.pad_token = tokenizer.eos_token` reuses the end-of-sequence token as the padding token.
- This is necessary because Gemma does not provide a separate padding token by default.
- `tokenizer.padding_side = "right"` means shorter examples are padded at the end rather than at the beginning.

### Project Impact
This cell ensures that training examples can be batched correctly even if they have different lengths. It improves the project by making tokenization and batching compatible with Gemma’s causal language modeling setup.


In [11]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set padding token (Gemma uses eos_token for padding)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token}")

Tokenizer loaded. Vocab size: 256000
Pad token: <eos>


## Cell 12 — Configure LoRA
Defines and applies the LoRA adapter setup.

- `LoraConfig(...)` specifies how the LoRA adapters should be created.
- `r=LORA_R` sets the low-rank dimension of the trainable matrices.
- Instead of training a very large full matrix, LoRA trains two much smaller matrices.
- `lora_alpha=LORA_ALPHA` scales the contribution of the LoRA updates.
- `target_modules=TARGET_MODULES` restricts LoRA to specific attention layers such as `q_proj` and `v_proj`.
- `lora_dropout=LORA_DROPOUT` randomly drops some adapter activations during training to improve generalization.
- `bias="none"` means bias parameters are not trained.
- `task_type="CAUSAL_LM"` tells PEFT that the model is used for text generation.
- `get_peft_model(model, lora_config)` attaches the LoRA adapters to the base model.
- `model.print_trainable_parameters()` shows how many parameters remain trainable after applying PEFT.

### Project Impact
This cell is central to the project because it makes fine-tuning cheap and efficient. The model can learn domain-specific Austrian tax law patterns without updating all original weights.

In [12]:
# LoRA configuration
lora_config = LoraConfig(
    r=LORA_R,                       # rank of the low-rank matrices
    lora_alpha=LORA_ALPHA,          # scaling factor
    target_modules=TARGET_MODULES,  # which layers to adapt
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Show trainable parameters
model.print_trainable_parameters()

trainable params: 3,194,880 || all params: 2,617,536,768 || trainable%: 0.1221


## Cell 13 — Configure Training
Defining how the fine-tuning process should run and initializes the trainer.

- `SFTConfig(...)` stores the supervised fine-tuning settings.
- `output_dir` is the folder where training outputs and checkpoints are written.
- `num_train_epochs=NUM_EPOCHS` controls how many full passes over the training data are performed.
- `per_device_train_batch_size=BATCH_SIZE` defines how many examples are processed at once on one GPU.
- `gradient_accumulation_steps=GRAD_ACCUM_STEPS` increases the effective batch size without loading all examples at once.
- `learning_rate=LEARNING_RATE` controls the update size for trainable parameters.
- `weight_decay=0.01` regularizes the model and helps reduce overfitting.
- `warmup_steps=8` slowly increases the learning rate at the start of training.
- `lr_scheduler_type="cosine"` decreases the learning rate smoothly over time.
- `logging_steps=10` prints progress every ten training steps.
- `save_strategy="epoch"` saves the model after each epoch.
- `bf16=True` uses bfloat16 precision, which is well suited for Gemma 2.
- `optim="paged_adamw_8bit"` uses a memory-efficient optimizer.
- `report_to="none"` disables external experiment tracking.
- `SFTTrainer(...)` creates the trainer object that handles batching, tokenization, loss computation, and parameter updates automatically.
- `processing_class=tokenizer` means the trainer tokenizes text on its own during training.

### Project Impact
This cell defines the training behavior and strongly affects stability and final quality. It is where the project balances GPU limits, runtime, and generalization performance.


In [13]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="/kaggle/working/results",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=8,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("Trainer ready.")


Adding EOS to train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Trainer ready.


## Cell 14 — Start Training
Starts the supervised fine-tuning process.

- `trainer.train()` runs all configured epochs and batches.
- During training, the trainer computes the loss, updates the LoRA weights, logs progress, and saves checkpoints according to the configuration.
- On a Kaggle T4 GPU, this step can take roughly 25 to 30 minutes.

### Project Impact
This is the cell where the actual learning happens. Its output directly determines whether the base Gemma model becomes better adapted to the Austrian tax law task.

In [14]:
# Start training
trainer.train()
print("Training complete!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.


Step,Training Loss
10,3.783845
20,2.461770
30,1.899566
40,1.613339
50,1.496142
60,1.446475
70,1.404949


Training complete!


## Cell 15 — Merge and Save the Fine-Tuned Model
Merging the LoRA adapters into the base model and saves the result.

- `merge_and_unload()` combines the learned LoRA weights with the frozen base weights.
- After merging, the model can be loaded as one single model instead of a base model plus separate adapters.
- `save_pretrained(SAVE_DIR)` saves the merged model weights and configuration files.
- `tokenizer.save_pretrained(SAVE_DIR)` saves the matching tokenizer to the same directory.

### Project Impact
This cell makes the trained model reusable for later inference and submission. It improves the project workflow because loading a single merged model is simpler and less error-prone.


In [15]:
# Merge LoRA weights into base model
merged_model = model.merge_and_unload()

# Save merged model and tokenizer
os.makedirs(SAVE_DIR, exist_ok=True)
merged_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to: {SAVE_DIR}")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /kaggle/working/model2_gemma_qlora


## Cell 16 — Load the Test Dataset
Loads the question dataset that the fine-tuned model will answer during inference.

- `pd.read_csv(DATASET_CSV)` reads the CSV file containing all test questions into a DataFrame called `test_df`.
- `len(test_df)` prints the total number of questions.
- `list(test_df.columns)` shows the column names for a quick format check.
- `test_df.head()` displays the first few rows as a visual sanity check.

### Project Impact
This cell loads the input that drives the entire inference phase. If the dataset path is wrong or the file format has changed, all following prediction steps will fail.

In [16]:
# Load test questions
test_df = pd.read_csv(DATASET_CSV)
print(f"Test dataset: {len(test_df)} questions")
print(f"Columns: {list(test_df.columns)}")
test_df.head()

Test dataset: 643 questions
Columns: ['id', 'prompt']


,id,prompt
0,CORP-TAX-001,Was ist die steuerliche Bemessungsgrundlage fü...
1,CORP-TAX-002,"Welche steuerlichen Konsequenzen hat es, wenn ..."
2,CORP-TAX-003,"Welche Körperschaften sind verpflichtet, sämtl..."
3,CORP-TAX-004,Wie wird eine Dividende einer österreichischen...
4,CORP-TAX-005,Was unterscheidet die steuerliche Behandlung e...


## Cell 17 — Reload the Fine-Tuned Model for Inference
Reloads the saved fine-tuned model so it can be used for answer generation.

- `AutoModelForCausalLM.from_pretrained(SAVE_DIR, ...)` loads the merged model from disk.
- `quantization_config=bnb_config` applies 4-bit quantization again to reduce memory usage during inference.
- `device_map="auto"` lets Transformers choose the best available device placement automatically.
- `torch_dtype=torch.float16` keeps inference efficient by using half precision.
- `AutoTokenizer.from_pretrained(SAVE_DIR)` loads the tokenizer that matches the fine-tuned model.
- `pad_token = eos_token` sets the padding token, which is required for generation.

### Project Impact
This cell separates training from prediction and prepares the model for the actual benchmark task. The project output is based on inference with the saved, adapted model rather than the training session itself.

In [17]:
# Reload the fine-tuned model for inference (in case memory was freed)
inf_model = AutoModelForCausalLM.from_pretrained(
    SAVE_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

inf_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
inf_tokenizer.pad_token = inf_tokenizer.eos_token

print("Fine-tuned model loaded for inference.")

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Fine-tuned model loaded for inference.


## Cell 18+19 — Inference Loop with Checkpointing
Runs inference over the dataset and periodically stores progress.

- `os.path.exists(CHECKPOINT_FILE)` checks whether a checkpoint file already exists.
- If it exists, previously generated results are loaded so the notebook can resume instead of starting from zero.
- `json.load(f)` reads the checkpoint content from disk into Python.
- `start_idx = len(results)` stores how many answers have already been produced.
- `time.time()` returns the current time in seconds.
- `elapsed = time.time() - start_time` calculates how much time has passed.
- `rate = (idx + 1 - start_idx) / elapsed * 60` estimates the inference speed in questions per minute.
- `json.dump(results, f, ensure_ascii=False)` writes the current results back to disk as JSON.
- `ensure_ascii=False` preserves German characters such as `ä`, `ö`, and `ü` correctly.

### Project Impact
This cell makes large-scale generation safer and more practical on unstable cloud sessions. It is important for the project because losing hundreds of generated answers due to a session interruption would waste a lot of time.


In [18]:
def generate_answer(question, model, tokenizer):
    """Generate an answer for a single question using the fine-tuned model."""
    # Format prompt in Gemma chat template (only the user turn + start of model turn)
    prompt = (
        f"<start_of_turn>user\n{SYSTEM_MSG}\n{question}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate — greedy decoding (faster + deterministic for factual answers)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the new tokens (skip the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Remove any trailing turn markers
    answer = answer.replace("<end_of_turn>", "").strip()

    return answer

# Test with one question first
test_question = test_df["prompt"].iloc[0]
test_answer = generate_answer(test_question, inf_model, inf_tokenizer)
print(f"Q: {test_question}")
print(f"A: {test_answer[:300]}")

Q: Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?
A: Die steuerliche Bemessungsgrundlage für die Körperschaftsteuer in Österreich ist **die Gewinne des Unternehmens**. 

Hier sind einige wichtige Punkte:

* **Gewinn:** Die Bemessungsgrundlage umfasst alle Einkünfte, die das Unternehmen im Kalenderjahr erzielt hat, einschließlich der Einnahmen aus alle


In [19]:
import json
import time

# Check for existing checkpoint
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r") as f:
        results = json.load(f)
    start_idx = len(results)
    print(f"Resuming from checkpoint: {start_idx} already done.")
else:
    results = []
    start_idx = 0
    print("Starting fresh inference.")

total = len(test_df)
print(f"Total: {total} | Remaining: {total - start_idx}")

start_time = time.time()

for idx in range(start_idx, total):
    row = test_df.iloc[idx]

    try:
        answer = generate_answer(row["prompt"], inf_model, inf_tokenizer)
    except Exception as e:
        print(f"  ERROR at {row['id']}: {e}")
        answer = "Fehler bei der Generierung."

    results.append({"id": row["id"], "answer": answer})

    # Progress every 10 questions
    if (idx + 1) % 10 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1 - start_idx) / elapsed * 60
        print(f"  [{idx+1}/{total}] {rate:.1f} q/min | last: {row['id']}")

    # Checkpoint every 50 questions
    if (idx + 1) % 50 == 0:
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump(results, f, ensure_ascii=False)
        print(f"  >> Checkpoint saved at {idx+1} questions.")

print(f"\nInference complete. Generated {len(results)} answers.")

Starting fresh inference.
Total: 643 | Remaining: 643
  [10/643] 2.7 q/min | last: CORP-TAX-010
  [20/643] 2.7 q/min | last: CORP-TAX-020
  [30/643] 2.8 q/min | last: CORP-TAX-030
  [40/643] 2.8 q/min | last: CORP-TAX-040
  [50/643] 2.8 q/min | last: CORP-TAX-050
  >> Checkpoint saved at 50 questions.
  [60/643] 2.8 q/min | last: CORP-TAX-060
  [70/643] 2.8 q/min | last: TAX-INTL-010
  [80/643] 2.8 q/min | last: TAX-INTL-020
  [90/643] 2.8 q/min | last: TAX-INTL-030
  [100/643] 2.8 q/min | last: TAX-INTL-040
  >> Checkpoint saved at 100 questions.
  [110/643] 2.8 q/min | last: TAX-INTL-050
  [120/643] 2.8 q/min | last: TAX-INTL-060
  [130/643] 2.8 q/min | last: TAX-INTL-070
  [140/643] 2.9 q/min | last: TAX-INTL-080
  [150/643] 2.9 q/min | last: ESTG-NEW-010
  >> Checkpoint saved at 150 questions.
  [160/643] 2.8 q/min | last: ESTG-NEW-020
  [170/643] 2.9 q/min | last: VAT-DOM-010
  [180/643] 2.9 q/min | last: VAT-DOM-020
  [190/643] 2.9 q/min | last: VAT-DOM-030
  [200/643] 2.9 q/min 

## Cell 20 — Save the Final Results
This cell converts the generated answers into the final CSV output format.

- `pd.DataFrame(results)` turns the Python result list into a pandas DataFrame.
- `id_order = test_df["id"].tolist()` stores the original order of IDs from the input dataset.
- `set_index("id")` uses the ID column as the temporary row key.
- `.loc[id_order]` reorders the predictions so they match the original dataset exactly.
- `.reset_index()` turns the ID back into a normal column.
- `to_csv(OUTPUT_CSV, index=False)` saves the final file without adding an extra pandas index column.

### Project Impact
This cell produces the deliverable that will later be submitted and evaluated. Correct ordering and formatting are critical because even good answers are not useful if the CSV structure is wrong.

In [20]:
# Create output dataframe from results
result_df = pd.DataFrame(results)

# Reorder to match original input order
id_order = test_df["id"].tolist()
result_df = result_df.set_index("id").loc[id_order].reset_index()

# Save to CSV
result_df.to_csv(OUTPUT_CSV, index=False)
print(f"Results saved to: {OUTPUT_CSV}")
print(f"Shape: {result_df.shape}")
result_df.head()

Results saved to: /kaggle/working/model2_finetuned_results.csv
Shape: (643, 2)


,id,answer
0,CORP-TAX-001,Die steuerliche Bemessungsgrundlage für die Kö...
1,CORP-TAX-002,In Österreich hat die Vergabe eines zinslosen ...
2,CORP-TAX-003,In Österreich müssen folgende Körperschaften s...
3,CORP-TAX-004,## Dividendenbehandlung in Österreich: Tochter...
4,CORP-TAX-005,In Österreich unterscheiden sich die steuerlic...


## Cell 21 — Verify the Output
This cell checks whether the final output file is complete and correctly structured.

- `pd.read_csv(OUTPUT_CSV)` reloads the saved result file.
- `expected = len(test_df)` stores the expected number of rows based on the input dataset.
- `assert len(result) == expected` checks whether every question has exactly one answer.
- `assert list(result.columns) == ["id", "answer"]` verifies the required column names.
- `result["answer"].notna().all()` checks that no answer is missing (NaN).
- `result["answer"].str.strip().ne("").all()` checks that no answer is an empty string.
- `result["id"].is_unique` verifies that every question ID appears only once.
- If all checks pass, the notebook prints a success message with summary statistics.

### Project Impact
This final validation step protects the project from submission errors. It ensures that the fine-tuned model output is not only generated, but also complete, consistent, and ready for evaluation.

In [21]:
# Reload and verify the saved CSV
result = pd.read_csv(OUTPUT_CSV)
expected = len(test_df)  # dynamic — avoids hardcoding row count

assert len(result) == expected, f"Expected {expected} rows, got {len(result)}"
assert list(result.columns) == ["id", "answer"], f"Wrong columns: {list(result.columns)}"
assert result["answer"].notna().all(), "Some answers are NaN!"
assert result["answer"].str.strip().ne("").all(), "Some answers are empty strings!"
assert result["id"].is_unique, "Duplicate IDs found!"

print(f"Verification passed. {expected} answers ready.")
print(f"  Columns: {list(result.columns)}")
print(f"  Avg answer length: {result['answer'].str.len().mean():.0f} chars")

Verification passed. 643 answers ready.
  Columns: ['id', 'answer']
  Avg answer length: 1257 chars
